# Step 4A.8 — аудит довжини CVE description (перед фіксацією max_length для B3)
Не кодує нічого через SecBERT, не навчає жодної моделі. Лише вимірює:
1. Розподіл довжини `description` у СИМВОЛАХ (швидко, без токенізатора).
2. Розподіл довжини в ТОКЕНАХ SecBERT (повільніше, але точніше — саме це визначає
   реальне обрізання при inference).
3. Частку CVE, що обрізаються при max_length = 128 / 256 / 384 / 512.
4. Частку порожніх/дуже коротких description (для `description_missing`).

Ядро: IDS Research.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

CTI_DIR = Path("/srv/ids_research/data/cti")
ds = pd.read_parquet(CTI_DIR / "cve_forecast_dataset.parquet", columns=["cve_id", "description"])
print(f"Завантажено: {len(ds):,} CVE")

desc = ds["description"].fillna("")
is_empty = desc.str.strip().eq("")
print(f"\nПорожніх description: {int(is_empty.sum()):,} ({is_empty.mean():.2%})")
print(f"Непорожніх: {int((~is_empty).sum()):,}")

Завантажено: 379,036 CVE

Порожніх description: 0 (0.00%)
Непорожніх: 379,036


In [2]:
# ---------- 1. Розподіл довжини в СИМВОЛАХ (швидко, орієнтовний) ----------
char_len = desc.str.len()
print("Довжина description у символах (усі рядки, включно з порожніми):")
print(char_len.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

print("\nТе саме, лише серед НЕПОРОЖНІХ:")
print(char_len[~is_empty].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

Довжина description у символах (усі рядки, включно з порожніми):
count    379036.000000
mean        346.557696
std         339.413106
min          10.000000
50%         259.000000
75%         394.000000
90%         609.000000
95%         813.000000
99%        1772.650000
max        3998.000000
Name: description, dtype: float64

Те саме, лише серед НЕПОРОЖНІХ:
count    379036.000000
mean        346.557696
std         339.413106
min          10.000000
50%         259.000000
75%         394.000000
90%         609.000000
95%         813.000000
99%        1772.650000
max        3998.000000
Name: description, dtype: float64


In [3]:
# ---------- 2. Розподіл довжини в ТОКЕНАХ SecBERT (точний, повільніший — на вибірці) ----------
# Токенізація всіх 379,036 CVE зайняла б багато часу лише для аудиту; беремо репрезентативну
# випадкову вибірку СЕРЕД НЕПОРОЖНІХ описів (не стратифіковану — просто випадкову; порожні
# описи виключені окремо, бо вони дають тривіальні 2 токени і спотворили б розподіл).
import sys, subprocess, importlib.util
if importlib.util.find_spec("transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)
from transformers import AutoTokenizer

MODEL_NAME = "jackaduma/SecBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

SAMPLE_SIZE = 5000
non_empty_desc = desc[~is_empty]
sample = non_empty_desc.sample(n=min(SAMPLE_SIZE, len(non_empty_desc)), random_state=42)

print(f"Токенізую вибірку з {len(sample):,} непорожніх description (без спеціальних токенів "
      f"для чистого підрахунку довжини тексту)...")
token_lengths = sample.map(lambda t: len(tokenizer.encode(t, add_special_tokens=False)))

print("\nРозподіл довжини в ТОКЕНАХ (без [CLS]/[SEP], на вибірці):")
print(token_lengths.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

Токенізую вибірку з 5,000 непорожніх description (без спеціальних токенів для чистого підрахунку довжини тексту)...

Розподіл довжини в ТОКЕНАХ (без [CLS]/[SEP], на вибірці):
count    5000.000000
mean       80.238400
std       109.361607
min         4.000000
50%        56.000000
75%        86.000000
90%       137.000000
95%       188.000000
99%       504.060000
max      2069.000000
Name: description, dtype: float64


In [4]:
# ---------- 3. Частка CVE, що ОБРІЗАЮТЬСЯ при різних max_length (+2 для [CLS]/[SEP]) ----------
print("Частка (на вибірці) description, довших за max_length (тобто будуть обрізані):")
for max_len in [128, 256, 384, 512]:
    effective_limit = max_len - 2   # мінус [CLS] і [SEP]
    truncated_share = (token_lengths > effective_limit).mean()
    print(f"  max_length={max_len:4d}: обрізається {truncated_share:.2%} description "
          f"(поріг у токенах тексту: {effective_limit})")

Частка (на вибірці) description, довших за max_length (тобто будуть обрізані):
  max_length= 128: обрізається 11.70% description (поріг у токенах тексту: 126)
  max_length= 256: обрізається 2.76% description (поріг у токенах тексту: 254)
  max_length= 384: обрізається 1.38% description (поріг у токенах тексту: 382)
  max_length= 512: обрізається 0.98% description (поріг у токенах тексту: 510)


In [5]:
# ---------- 4. Дуже короткі (не порожні, але майже) description ----------
print("Розподіл ДУЖЕ коротких (1-20 символів) непорожніх description — межові випадки:")
very_short = char_len[(~is_empty) & (char_len <= 20)]
print(f"Кількість: {len(very_short):,} ({len(very_short)/len(ds):.3%} від усіх CVE)")
if len(very_short):
    print("\nПриклади:")
    print(desc[(~is_empty) & (char_len <= 20)].head(10).tolist())

Розподіл ДУЖЕ коротких (1-20 символів) непорожніх description — межові випадки:
Кількість: 5 (0.001% від усіх CVE)

Приклади:
['NFS cache poisoning.', 'Denial of service', 'Multiple CWE-78', 'Vulnerability Title', '## Summary']


In [6]:
# ---------- ПІДСУМОК: рекомендація для max_length ----------{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Step 4A.9 — B3: масове кодування 379,036 CVE через SecBERT (embedding generation ONLY)\n",
    "Жодного навчання classifier тут немає — лише encoder → embeddings → збереження + перевірка.\n",
    "Розділення embedding generation і classifier training навмисне: якщо classifier покаже\n",
    "нестабільність пізніше, можна аналізувати representation окремо від classifier.\n",
    "\n",
    "**B3 spec (заморожено):**\n",
    "- Encoder: `jackaduma/SecBERT`\n",
    "- `max_length=256` (97.24% описів на аудит-вибірці без обрізання, Step 4A.8 — не всі 379,036 CVE перевірялись поштучно)\n",
    "- Pooling: attention-mask weighted mean (лише реальні токени, не [PAD])\n",
    "- L2 normalization\n",
    "- `description_missing` НЕ додається (0/379,036 missing у frozen dataset)\n",
    "- Дуже короткі description кодуються як звичайний текст, без спецобробки\n",
    "\n",
    "Ядро: IDS Research."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import sys, subprocess, importlib.util\n",
    "if importlib.util.find_spec(\"transformers\") is None:\n",
    "    subprocess.run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"transformers\"], check=True)\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import torch\n",
    "import torch.nn.functional as F\n",
    "from pathlib import Path\n",
    "from transformers import AutoTokenizer, AutoModel\n",
    "\n",
    "CTI_DIR = Path(\"/srv/ids_research/data/cti\")\n",
    "RESULTS_DIR = Path.home() / \"notebooks\" / \"2 article\" / \"release\" / \"results\"\n",
    "RESULTS_DIR.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "MODEL_NAME = \"jackaduma/SecBERT\"\n",
    "MAX_LENGTH = 256\n",
    "BATCH_SIZE = 32\n",
    "EXPECTED_N_ROWS = 379036\n",
    "EMBEDDING_DIM = 768\n",
    "\n",
    "device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n",
    "print(f\"device={device}, MAX_LENGTH={MAX_LENGTH}, BATCH_SIZE={BATCH_SIZE}\")"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "# ---------- Завантаження frozen dataset + перевірка розміру ----------\n",
    "ds = pd.read_parquet(CTI_DIR / \"cve_forecast_dataset.parquet\", columns=[\"cve_id\", \"description\"])\n",
    "assert len(ds) == EXPECTED_N_ROWS, f\"actual={len(ds)}, expected={EXPECTED_N_ROWS} — dataset змінився!\"\n",
    "assert ds[\"cve_id\"].is_unique, \"cve_id не унікальні — щось не так із frozen dataset\"\n",
    "print(f\"✅ {len(ds):,} рядків, cve_id унікальні\")\n",
    "\n",
    "n_empty = ds[\"description\"].fillna(\"\").str.strip().eq(\"\").sum()\n",
    "assert n_empty == 0, (\n",
    "    f\"Знайдено {n_empty} порожніх description — frozen dataset НЕ відповідає Step 4A.8! \"\n",
    "    f\"description_missing НЕ реалізовано в цьому ноутбуці навмисне (рішення Step 4A.8: \"\n",
    "    f\"0/379,036 missing) — якщо ця перевірка впала, спершу треба переглянути рішення, \"\n",
    "    f\"а не просто продовжувати кодування.\"\n",
    ")\n",
    "print(f\"✅ Порожніх description: {n_empty}\")"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "# ---------- Завантаження SecBERT (СУВОРО локально, без звернення до інтернету) ----------\n",
    "# local_files_only=True критично для відтворюваності: цей крок НЕ повинен непомітно\n",
    "# підтягнути іншу версію файлів чи залежати від доступності HuggingFace у момент запуску.\n",
    "# Модель уже закешована локально зі Step 4A.7 — якщо цей рядок впаде з помилкою \"not found\n",
    "# locally\", значить кеш відсутній/пошкоджений, і це треба розібрати, а не мовчки качати заново.\n",
    "tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)\n",
    "model = AutoModel.from_pretrained(MODEL_NAME, local_files_only=True).to(device)\n",
    "model.eval()\n",
    "print(f\"Модель завантажена ЛОКАЛЬНО (local_files_only=True). Параметрів: \"\n",
    "      f\"{sum(p.numel() for p in model.parameters()):,}\")"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "# ---------- Masked mean pooling + L2 normalization (batch inference) ----------\n",
    "def masked_mean_pooling(last_hidden_state, attention_mask):\n",
    "    \"\"\"Mean pooling ЛИШЕ по реальних токенах (attention_mask=1), [PAD] не впливає на середнє.\"\"\"\n",
    "    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()\n",
    "    summed = torch.sum(last_hidden_state * mask, dim=1)\n",
    "    counts = torch.clamp(mask.sum(dim=1), min=1e-9)\n",
    "    return summed / counts\n",
    "\n",
    "def encode_batch(texts):\n",
    "    inputs = tokenizer(texts, return_tensors=\"pt\", truncation=True, max_length=MAX_LENGTH,\n",
    "                       padding=True).to(device)\n",
    "    with torch.inference_mode():   # чистіше й дещо ощадливіше за пам'яттю, ніж no_grad(), для pure inference\n",
    "        output = model(**inputs)\n",
    "    emb = masked_mean_pooling(output.last_hidden_state, inputs[\"attention_mask\"])\n",
    "    emb = F.normalize(emb, p=2, dim=1)   # L2 normalization\n",
    "    return emb.cpu().numpy()\n",
    "\n",
    "# швидка перевірка на 3 прикладах перед повним прогоном\n",
    "test_emb = encode_batch(ds[\"description\"].iloc[:3].tolist())\n",
    "print(f\"Тестовий batch: shape={test_emb.shape}, норма кожного вектора (має бути ~1.0 після L2): \"\n",
    "      f\"{np.linalg.norm(test_emb, axis=1)}\")"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "# ---------- Повне кодування 379,036 CVE, батчами, зі збереженням прогресу ----------\n",
    "import time\n",
    "\n",
    "texts = ds[\"description\"].tolist()\n",
    "n = len(texts)\n",
    "all_embeddings = np.zeros((n, EMBEDDING_DIM), dtype=np.float32)\n",
    "\n",
    "t0 = time.time()\n",
    "for start in range(0, n, BATCH_SIZE):\n",
    "    end = min(start + BATCH_SIZE, n)\n",
    "    batch_emb = encode_batch(texts[start:end])\n",
    "    all_embeddings[start:end] = batch_emb\n",
    "    if (start // BATCH_SIZE) % 200 == 0:\n",
    "        elapsed = time.time() - t0\n",
    "        processed = end   # фактично оброблено, а не start+BATCH_SIZE (некоректно на останньому неповному батчі)\n",
    "        rate = processed / max(elapsed, 1e-9)\n",
    "        eta = (n - processed) / max(rate, 1e-9)\n",
    "        print(f\"  {end:,}/{n:,}  ({elapsed:.0f}с, ~{rate:.1f} CVE/с, ETA ~{eta/60:.1f} хв)\")\n",
    "\n",
    "total_time = time.time() - t0\n",
    "print(f\"\\n✅ Кодування завершено за {total_time/60:.1f} хв ({n/total_time:.1f} CVE/с)\")"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "# ---------- Перевірки якості embeddings ----------\n",
    "print(\"1. Кількість embeddings == 379,036?\")\n",
    "assert all_embeddings.shape[0] == EXPECTED_N_ROWS\n",
    "print(f\"   ✅ {all_embeddings.shape[0]:,}\")\n",
    "\n",
    "print(\"\\n2. Dimension == 768?\")\n",
    "assert all_embeddings.shape[1] == EMBEDDING_DIM\n",
    "print(f\"   ✅ {all_embeddings.shape[1]}\")\n",
    "\n",
    "print(\"\\n3. NaN?\")\n",
    "n_nan = np.isnan(all_embeddings).sum()\n",
    "assert n_nan == 0, f\"Знайдено {n_nan} NaN-значень!\"\n",
    "print(f\"   ✅ 0\")\n",
    "\n",
    "print(\"\\n4. Inf?\")\n",
    "n_inf = np.isinf(all_embeddings).sum()\n",
    "assert n_inf == 0, f\"Знайдено {n_inf} Inf-значень!\"\n",
    "print(f\"   ✅ 0\")\n",
    "\n",
    "print(\"\\n5. cve_id унікальні (повторна перевірка)?\")\n",
    "assert ds[\"cve_id\"].is_unique\n",
    "print(f\"   ✅ так\")\n",
    "\n",
    "print(\"\\n6. Порядок CVE відповідає dataset (embeddings[i] відповідає ds.iloc[i])?\")\n",
    "# Перевіряємо непрямо: перекодовуємо СПРАВДІ ВИПАДКОВІ (не фіксовані) індекси і звіряємо\n",
    "# з масивом — детермінований seed для відтворюваності запуску, але вибірка реально випадкова.\n",
    "rng = np.random.default_rng(42)\n",
    "sample_idx = rng.choice(n, size=5, replace=False).tolist()\n",
    "recheck_texts = [texts[i] for i in sample_idx]\n",
    "recheck_emb = encode_batch(recheck_texts)\n",
    "matches = [np.allclose(all_embeddings[i], recheck_emb[j], atol=1e-5) for j, i in enumerate(sample_idx)]\n",
    "assert all(matches), f\"Розбіжність порядку на індексах {sample_idx}: {matches}\"\n",
    "print(f\"   ✅ перевірено на {len(sample_idx)} випадкових індексах: {sample_idx}\")\n",
    "\n",
    "print(\"\\n7. Норма векторів ~1.0 (L2 normalization застосована)?\")\n",
    "norms = np.linalg.norm(all_embeddings, axis=1)\n",
    "print(f\"   min={norms.min():.4f}, max={norms.max():.4f}, mean={norms.mean():.4f}\")\n",
    "assert np.allclose(norms, 1.0, atol=1e-3), \"L2-норма відхиляється від 1.0\"\n",
    "print(\"   ✅\")\n",
    "\n",
    "print(\"\\n✅ Усі перевірки embeddings пройдено.\")"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "# ---------- Збереження: embeddings (.npy) + cve_id mapping (.parquet) ----------\n",
    "emb_path = RESULTS_DIR / \"step4a9_b3_embeddings.npy\"\n",
    "mapping_path = RESULTS_DIR / \"step4a9_b3_cve_id_mapping.parquet\"\n",
    "\n",
    "np.save(emb_path, all_embeddings)\n",
    "ds[[\"cve_id\"]].reset_index(drop=True).to_parquet(mapping_path, index=False)\n",
    "\n",
    "print(f\"Збережено:\")\n",
    "print(f\"  {emb_path} — {all_embeddings.nbytes / 1e6:.1f} МБ, shape={all_embeddings.shape}\")\n",
    "print(f\"  {mapping_path} — cve_id у ТОМУ Ж порядку, що й рядки в .npy (index i <-> cve_id[i])\")\n",
    "print(f\"\\nℹ Завантаження для подальшого використання (Step 4A.10, B3 classifier):\")\n",
    "print(f\"  embeddings = np.load('{emb_path}')\")\n",
    "print(f\"  cve_id_order = pd.read_parquet('{mapping_path}')['cve_id']\")\n",
    "print(f\"  # embeddings[i] відповідає cve_id_order[i]\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
print("=" * 70)
print("ПІДСУМОК")
print("=" * 70)
p95 = token_lengths.quantile(0.95)
p99 = token_lengths.quantile(0.99)
print(f"P95 довжини в токенах (вибірка): {p95:.0f}")
print(f"P99 довжини в токенах (вибірка): {p99:.0f}")
print(f"\nРекомендація: обрати НАЙМЕНШЕ значення max_length із {{128,256,384,512}}, що покриває")
print(f"  ≥95% description без обрізання (дивіться таблицю з кроку 3 вище) — компроміс між")
print(f"  повнотою тексту і швидкістю масового кодування 379,036 CVE.")
print(f"\nПорожніх description: {is_empty.mean():.2%} — для них: zero-vector embedding +")
print(f"  description_missing=1 (НЕ інференс SecBERT на порожньому/пробільному рядку).")

ПІДСУМОК
P95 довжини в токенах (вибірка): 188
P99 довжини в токенах (вибірка): 504

Рекомендація: обрати НАЙМЕНШЕ значення max_length із {128,256,384,512}, що покриває
  ≥95% description без обрізання (дивіться таблицю з кроку 3 вище) — компроміс між
  повнотою тексту і швидкістю масового кодування 379,036 CVE.

Порожніх description: 0.00% — для них: zero-vector embedding +
  description_missing=1 (НЕ інференс SecBERT на порожньому/пробільному рядку).
